# The Category That Only Exists in Production

**Topic:** Categorical encoding &nbsp;·&nbsp; **Difficulty:** Medium

A merchant-ID column is drawn from 3,000 possible merchants, most of them rare.
In production, new merchants sign up daily, so the encoder meets identifiers it
has never seen, and a naive lookup raises `KeyError` at 3am.

Build an encoder that survives this: frequency encoding, with rare training
levels and unseen test levels both folded into a single `__other__` bucket.

## The data (created in the next cell)

- `train_ids` `(20000,)` and `test_ids` `(5000,)`: merchant ID strings.
- About 12% of test rows carry a brand-new ID, and in total roughly 16% of test
  rows carry an ID that never appears in training.
- The ID frequency distribution is long-tailed: of the merchants that appear in training, most appear only once or twice.

## Your task

With **NumPy and pandas**:

1. Count how often each ID appears in **training**.
2. Any training ID appearing fewer than **5** times is rare: it maps to the
   `__other__` bucket, whose count is the total of all rare IDs.
3. Encode both splits as the (training) count of their bucket. Unseen test
   IDs also map to `__other__`.
4. Set `answers` to a dict with `"n_kept_levels"` (int, distinct IDs kept as
   themselves), `"other_count"` (int, training rows in `__other__`),
   `"n_unseen_test"` (int, test rows whose ID never appeared in training), and
   `"test_encoded"`, the length-5000 encoded test vector.

## Evaluation

Counts must match exactly, and the encoded vector to a relative tolerance of 1e-6.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(202)
n_train, n_test = 20000, 5000
pop = np.array([f"m{i:05d}" for i in range(3000)])
weights = 1.0 / (np.arange(1, 3001) ** 1.15)
weights /= weights.sum()
train_ids = rng.choice(pop, n_train, p=weights)
seen_part = rng.choice(pop, int(n_test * 0.88), p=weights)
fresh = np.array([f"new{i:04d}" for i in range(n_test - len(seen_part))])
test_ids = rng.permutation(np.concatenate([seen_part, fresh]))
print(f"train {train_ids.shape} ({len(set(train_ids))} distinct), test {test_ids.shape}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("n_kept_levels", [], 395.0, -66.13801585172716, 1e-06),
    ("other_count", [], 2576.0, -431.32032616214985, 1e-06),
    ("n_unseen_test", [], 788.0, -131.94115567382534, 1e-06),
    ("test_encoded", [5000], 7913231.0, -287455.9118432704, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

`value_counts` gives the training frequencies, and a threshold splits them into kept
and rare. Encoding with `map` then `fillna(other_count)` handles unseen and rare
levels in one step: the `NaN` that a missing key produces is exactly the signal
you want to catch. Folding rare and unseen into the same bucket is what makes
train and test consistent.

```python
import numpy as np
import pandas as pd

tr = pd.Series(train_ids)
counts = tr.value_counts()

kept = counts[counts >= 5]
other_count = int(counts[counts < 5].sum())

lookup = kept.astype(float)
test_encoded = pd.Series(test_ids).map(lookup).fillna(float(other_count)).to_numpy()

answers = {
    "n_kept_levels": int(len(kept)),
    "other_count": other_count,
    "n_unseen_test": int((~pd.Series(test_ids).isin(set(train_ids))).sum()),
    "test_encoded": test_encoded,
}
```